<a href="https://colab.research.google.com/github/harshdeep3/Recommendation_system/blob/main/Recommendation_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exmaple Recommendation system using KNN

In [1]:
!which python

/usr/local/bin/python


In [2]:
!pip3 install virtualenv  # Install Virtualenv:
!virtualenv venv # Create a Virtual Environment:
!source /content/theanoEnv/bin/activate;  # Activate the Virtual Environment:
!pip install numpy
!pip install pandas
!pip install scikit-learn
!pip install seaborn


created virtual environment CPython3.10.12.final.0-64 in 776ms
  creator CPython3Posix(dest=/content/venv, clear=False, no_vcs_ignore=False, global=False)
  seeder FromAppData(download=False, pip=bundle, setuptools=bundle, wheel=bundle, via=copy, app_data_dir=/root/.local/share/virtualenv)
    added seed packages: pip==24.0, setuptools==69.1.0, wheel==0.42.0
  activators BashActivator,CShellActivator,FishActivator,NushellActivator,PowerShellActivator,PythonActivator
/bin/bash: line 1: /content/theanoEnv/bin/activate: No such file or directory


In [3]:
# required libraries
import numpy as np
import pandas as pd
import sklearn
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# Now, we create user-item matrix using scipy csr matrix
from scipy.sparse import csr_matrix

In [4]:
# load data
def load_data(file1, file2):
  """
  This loads the data for two given files
  """

  # loading movie dataset -> movie titles
  ratings = pd.read_csv(file1)

  # getting rattings data based on the movie id
  movies = pd.read_csv(file2)

  return movies, ratings

In [5]:
movies, ratings = load_data("https://s3-us-west-2.amazonaws.com/recommender-tutorial/ratings.csv", "https://s3-us-west-2.amazonaws.com/recommender-tutorial/movies.csv")

movies.head(), ratings.head()

(   movieId                               title  \
 0        1                    Toy Story (1995)   
 1        2                      Jumanji (1995)   
 2        3             Grumpier Old Men (1995)   
 3        4            Waiting to Exhale (1995)   
 4        5  Father of the Bride Part II (1995)   
 
                                         genres  
 0  Adventure|Animation|Children|Comedy|Fantasy  
 1                   Adventure|Children|Fantasy  
 2                               Comedy|Romance  
 3                         Comedy|Drama|Romance  
 4                                       Comedy  ,
    userId  movieId  rating  timestamp
 0       1        1     4.0  964982703
 1       1        3     4.0  964981247
 2       1        6     4.0  964982224
 3       1       47     5.0  964983815
 4       1       50     5.0  964982931)

In [6]:

# Find Lowest and Highest rated movies:
mean_rating = ratings.groupby('movieId')[['rating']].mean()
# Lowest rated movies
lowest_rated = mean_rating['rating'].idxmin()
movies.loc[movies['movieId'] == lowest_rated]
# Highest rated movies
highest_rated = mean_rating['rating'].idxmax()
movies.loc[movies['movieId'] == highest_rated]
# show number of people who rated movies rated movie highest
ratings[ratings['movieId']==highest_rated]
# show number of people who rated movies rated movie lowest
ratings[ratings['movieId']==lowest_rated]

## the above movies has very low dataset. We will use bayesian average
movie_stats = ratings.groupby('movieId')[['rating']].agg(['count', 'mean'])
movie_stats.columns = movie_stats.columns.droplevel()

In [7]:
def create_matrix(df):

  N = len(df['userId'].unique())
  M = len(df['movieId'].unique())

  # Map Ids to indices
  user_mapper = dict(zip(np.unique(df["userId"]), list(range(N))))
  movie_mapper = dict(zip(np.unique(df["movieId"]), list(range(M))))

  # Map indices to IDs
  user_inv_mapper = dict(zip(list(range(N)), np.unique(df["userId"])))
  movie_inv_mapper = dict(zip(list(range(M)), np.unique(df["movieId"])))

  user_index = [user_mapper[i] for i in df['userId']]
  movie_index = [movie_mapper[i] for i in df['movieId']]

  X = csr_matrix((df["rating"], (movie_index, user_index)), shape=(M, N))

  return X, user_mapper, movie_mapper, user_inv_mapper, movie_inv_mapper

X, user_mapper, movie_mapper, user_inv_mapper, movie_inv_mapper = create_matrix(ratings)


In [8]:
from sklearn.neighbors import NearestNeighbors

def find_similar_movies(movie_id, movie_data_matrix, num_neighbors=10, similarity_metric='cosine', return_distances=False):
    """
    This function finds the most similar movies to a given movie based on a provided data matrix.

    Args:
        movie_id (int): The ID of the movie to find similar movies for.
        movie_data_matrix (numpy.ndarray): A 2D array where rows represent movies and columns represent features used for similarity calculation.
        num_neighbors (int, optional): The number of most similar movies to return. Defaults to 10.
        similarity_metric (str, optional): The metric used to calculate similarity between movies. Defaults to 'cosine'.
        return_distances (bool, optional): Whether to return the distances to the neighboring movies along with their IDs. Defaults to False.

    Returns:
        list[int]: A list of movie IDs for the most similar movies to the input movie.
    """

    # Get the movie index within the data matrix
    movie_index = movie_mapper[movie_id]
    # Get the feature vector representing the movie
    movie_vector = movie_data_matrix[movie_index]

    # Create a Nearest Neighbors model with specified parameters
    knn = NearestNeighbors(n_neighbors=num_neighbors + 1, algorithm="brute", metric=similarity_metric)
    knn.fit(movie_data_matrix)

    # Reshape the movie vector for compatibility with kNN
    movie_vector = movie_vector.reshape(1, -1)

    # Find the nearest neighbors (including the input movie itself)
    neighbors = knn.kneighbors(movie_vector, return_distance=return_distances)

    # Extract movie IDs from the neighbors (excluding the input movie)
    similar_movie_ids = []
    for i in range(1, num_neighbors + 1):
        neighbor_id = neighbors.item(i)
        similar_movie_ids.append(movie_inv_mapper[neighbor_id])

    return similar_movie_ids

In [9]:
# This dictionary stores movie titles as key-value pairs, where the key is the movie ID and the value is the title.
movie_titles = dict(zip(movies['movieId'], movies['title']))

# Assign the movie ID you want to use as reference for recommendations (replace 3 with the actual ID)
movie_id = 3

similar_ids = find_similar_movies(movie_id, X, num_neighbors=10)

# Retrieve the title of the reference movie from the dictionary using its ID
movie_title = movie_titles[movie_id]

# Print a message mentioning the reference movie title
print(f"Since you watched {movie_title}")

# Loop through the list of similar movie IDs
for i in similar_ids:
  # Retrieve the title of each similar movie from the dictionary using its ID
  similar_movie_title = movie_titles[i]
  # Print the title of the similar movie
  print(similar_movie_title)

Since you watched Grumpier Old Men (1995)
Grumpy Old Men (1993)
Striptease (1996)
Nutty Professor, The (1996)
Twister (1996)
Father of the Bride Part II (1995)
Broken Arrow (1996)
Bio-Dome (1996)
Truth About Cats & Dogs, The (1996)
Sabrina (1995)
Birdcage, The (1996)


In [16]:
def recommend_movies_for_user(user_id, X, user_mapper, movie_mapper, movie_inv_mapper, k=10):
  """
  This function recommends movies for a user based on their highest rated movie.

  Args:
      user_id (int): The user ID to recommend movies for.
      ratings_data_frame (pandas.DataFrame): A DataFrame containing user-movie ratings data.
      user_mapper (dict, optional): A dictionary mapping user IDs to their corresponding information (if used).
      movie_mapper (dict, optional): A dictionary mapping movie IDs to their corresponding information (if used).
      movie_inv_mapper (dict, optional): An inverted dictionary mapping movie titles/information to their IDs (if used).
      k (int, optional): The number of similar movies to recommend (default 10).

  Returns:
      None
  """

  # Filter ratings data for the specific user
  df1 = ratings[ratings['userId'] == user_id]

  # Check if user exists in the data
  if df1.empty:
    print(f"User with ID {user_id} does not exist.")
    return

  # Find the movie ID with the highest rating for this user
  movie_id = df1[df1['rating'] == max(df1['rating'])]['movieId'].iloc[0]

  # Create a dictionary to map movie IDs to titles (if movie data is available)
  movie_titles = dict(zip(movies['movieId'], movies['title']))

  # Find similar movies based on the highest rated movie
  similar_ids = find_similar_movies(movie_id, X, k)

  # Get the title of the highest rated movie (handle potential missing movie data)
  movie_title = movie_titles.get(movie_id, "Movie not found")

  # Check if the highest rated movie was found
  if movie_title == "Movie not found":
    print(f"Movie with ID {movie_id} not found.")
    return

  print(f"Since you watched {movie_title}, you might also like:")

  # Recommend similar movies based on their IDs (handle potential missing movie data)
  for i in similar_ids:
    print(movie_titles.get(i, "Movie not found"))

In [17]:

user_id = 150

recommend_movies_for_user(
  user_id,
  X,
  user_mapper,
  movie_mapper,
  movie_inv_mapper,
  k=10
)

Since you watched Twelve Monkeys (a.k.a. 12 Monkeys) (1995), you might also like:
Pulp Fiction (1994)
Terminator 2: Judgment Day (1991)
Independence Day (a.k.a. ID4) (1996)
Seven (a.k.a. Se7en) (1995)
Fargo (1996)
Fugitive, The (1993)
Usual Suspects, The (1995)
Jurassic Park (1993)
Star Wars: Episode IV - A New Hope (1977)
Heat (1995)


In [18]:
user_id = 2300 # Replace with the desired user ID
recommend_movies_for_user(user_id, X, user_mapper, movie_mapper, movie_inv_mapper, k=10)


User with ID 2300 does not exist.
